In [ ]:
from datetime import time

# import torch
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

import sys
sys.path.append('..')

from pathlib import Path

In [ ]:
from tools.geometry import generate_detector
from tools.utils import generate_random_params
from tools.utils import load_single_event, save_single_event
import jax
import jax.numpy as jnp
from tools.simulation import setup_event_simulator

# Generate and save a single event
key = jax.random.PRNGKey(6)

detector_params = (
    jnp.array(50),           # scattering_length
    jnp.array(0.0),         # reflection_rate
    jnp.array(1000),         # absorption_length
    jnp.array(0.001)        # gumbel_softmax_temp
)


track_params = (
    jnp.array(500.0, dtype=jnp.float32),              # energy 
    jnp.array([0.0, 0.0, 0.0], dtype=jnp.float32),    # position
    jnp.array([jnp.pi/3, jnp.pi/4], dtype=jnp.float32)  # angles (theta, phi)
    #jnp.array([jnp.pi, jnp.pi/2], dtype=jnp.float32)  # angles (theta, phi)
)

In [ ]:
detector_names = ['HK']#, 'WCTE', 'IWCD', 'SK', 'HK', 'JUNO']

In [ ]:
generate_event = True
use_calibration = False

if generate_event:
    for name in detector_names:
        json_filename = f'../config/{name}_geom_config.json'
        detector = generate_detector(json_filename)
        detector_points = jnp.array(detector.all_points)
        Nphot = 1_000_000
        temperature = 0.0
        generate_event = False
        
        detector_type= None
        if name == 'TAO' or name == 'JUNO':
            detector_type='Sphere'
        elif 'Box' in name:
            detector_type='Box'
        else:
            detector_type='Cylinder'
        

        simulator = None
        single_event = None
        if use_calibration is False:
            simulator = setup_event_simulator(json_filename, Nphot, temperature=temperature, K=6, is_data=False, is_calibration=False, detector_type=detector_type, max_sensors_per_cell=10)
            single_event = jax.lax.stop_gradient(simulator(track_params, detector_params, key))
        else:
            source_params = (
                jnp.array([0.0, 0.0, 0.0], dtype=jnp.float32),
                jnp.array(1.0, dtype=jnp.float32)
            )
            simulator = setup_event_simulator(json_filename, Nphot, temperature=temperature, K=2, is_data=False, is_calibration=True, detector_type=detector_type, max_sensors_per_cell=10)
            single_event = jax.lax.stop_gradient(simulator(source_params, detector_params, key))

        # Create events folder if it doesn't exist
        events_dir = Path('../events')
        events_dir.mkdir(parents=True, exist_ok=True)
    
        save_single_event(single_event, track_params, detector_params, filename=f'../events/{name}_event_data.h5', calibration_mode=False)

In [ ]:
figures_dir = Path('figures')
figures_dir.mkdir(parents=True, exist_ok=True)

def visualize_3D_event_for_detector(name, colorscale='viridis', surface_color='gray'):
    _, _, indices, charges, times = load_single_event(f'../events/{name}_event_data.h5', None, calibration_mode=False)
    json_filename = f'../config/{name}_geom_config.json'
    detector = generate_detector(json_filename)
    figname = f'figures/{name}_3D_evt_display.pdf'
    detector.visualize_event_data_plotly_discs(indices, charges, times, show_all_sensors=True, log_scale=True, show_colorbar=False, dark_theme=False, plot_time=False, colorscale=colorscale, surface_color=surface_color, figname=figname)

In [ ]:
def check_missing_sensors(name):
    _, _, indices, charges, times = load_single_event(f'../events/{name}_event_data.h5', None, calibration_mode=False)
    json_filename = f'../config/{name}_geom_config.json'
    detector = generate_detector(json_filename)
    if len(detector.all_points) == len(indices):
        print('Success!')
    else:
        print('Sensors Missing:')
        print(len(detector.all_points), len(indices))

for name in detector_names:
    check_missing_sensors(name)

In [ ]:
_, _, indices, charges, times = load_single_event(f'../events/{"HK"}_event_data.h5', None, calibration_mode=False)

In [ ]:
len(indices)

In [ ]:
visualize_3D_event_for_detector('HK', surface_color='black', colorscale='inferno')

In [ ]:
# a simpler method to see the photosensor placements

import matplotlib.pyplot as plt

name = 'MidBox'
json_filename = f'../config/{name}_geom_config.json'
detector = generate_detector(json_filename)

detector = generate_detector(json_filename)
detector_points = jnp.array(detector.all_points)
photosensor_radius = detector.S_radius
#sphere_radius = detector.r

fig = plt.figure()
ax = fig.add_subplot(projection='3d')

ax.scatter(detector.all_points[:,0],detector.all_points[:,1],detector.all_points[:,2], s=0.05)